# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`  
This notebook demonstrates how to load and explore the FAIR<sup>2</sup> protein mass spectrometry dataset using the `mlcroissant` library. We examine the Croissant schema, extract data from its record sets, and perform initial analysis.

### Dataset Source
The dataset is described using a Croissant schema available at a public URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get dataset metadata as a Python dict for pretty printing
metadata_json = json.loads(dataset.metadata.to_json())
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields they contain.

In [ ]:
# List all available record sets and their fields
record_sets = dataset.metadata.record_sets()
print(f"Found {len(record_sets)} record set(s):\n")
rs_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    rs_ids.append(rs['@id'])
    if 'field' in rs:
        print(f"  Fields:")
        # Each field is an object or a list of objects
        for f in rs['field'] if isinstance(rs['field'], list) else [rs['field']]:
            f_id = f.get('@id', f)
            f_name = f.get('name', 'N/A')
            print(f"    - Field @id: {f_id}\tname: {f_name}")
    print()
print("All record set ids:", rs_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We will use the RecordSet `@id` and the field names identified above.
You can select any valid `record_set_id` from the list above.

In [ ]:
# For this dataset, let's extract the records for the main protein abundance analysis RecordSet
# If more than one RecordSet is present, you can work with others similarly.

# Example RecordSet IDs found above:
# We'll auto-select the first one for demonstration.
main_record_set_id = rs_ids[0] if rs_ids else None
if not main_record_set_id:
    raise RuntimeError("No record set found in the dataset schema.")

# Load data from all record sets into dataframes
dataframes = {}
for record_set_id in rs_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"  - Loaded {len(df)} rows with columns: {list(df.columns)}")
    dataframes[record_set_id] = df

# Preview the first few columns and records for the main recordset
print(f"Columns for RecordSet '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some common data processing steps:
- Filter records based on a numeric field's value
- Normalize that field
- Group by a categorical field (if available)

**Note:** All fields (columns) referenced by their `@id`. Please refer to the printed columns from the previous step.


In [ ]:
# Choose fields. (Update if column names differ.)
df = dataframes[main_record_set_id]
columns = df.columns.tolist()
print(f"All columns available: {columns}")

# Try to automatically select a numeric field for demonstration
# Otherwise, update this manually to fit the field @id or name (case-insensitive)
import numpy as np
numeric_field = None
for col in columns:
    # Check if column can be cast to float
    try:
        pd.to_numeric(df[col].dropna().head(5))
        numeric_field = col
        break
    except Exception:
        continue
if numeric_field is None:
    raise RuntimeError("No numeric field found in this dataset. Please specify manually.")
print(f"Selected numeric_field for analysis: {numeric_field}")

# Filter, normalize, group
threshold = np.nanmean(pd.to_numeric(df[numeric_field], errors='coerce'))
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
print(f"Filtered records with '{numeric_field}' > {threshold:.3f}:")
print(filtered_df[[numeric_field]].head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to select a categorical field for grouping
group_field = None
for col in columns:
    if col != numeric_field and df[col].dtype == 'object' and df[col].nunique() < max(10, len(df) // 10):
        group_field = col
        break
# Fallback if no categorical field found
if not group_field:
    print("No good categorical group field found; skipping groupby example.")
else:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the numeric field distribution and its normalized values, and—if available—a group-wise comparison.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

plt.figure(figsize=(10,4))
sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10,4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
plt.title(f"Normalized {numeric_field} (filtered records)")
plt.xlabel(f"{numeric_field} normalized")
plt.ylabel("Count")
plt.show()

if 'grouped_df' in locals():
    plt.figure(figsize=(12,4))
    ax = sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    ax.set_title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load a dataset described using a Croissant schema
- Explore the dataset's structure via record sets and fields
- Extract records into pandas DataFrames
- Apply standard EDA steps, filtering and normalizing data
- Visualize data distributions

This approach is adaptable to any FAIR dataset described with the Croissant standard, provided you reference all columns, fields, and sets by their `@id`s for reproducibility.